In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [6]:
from src.pipeline import (
    extract_data,
    validate_data,
    clean_sales_data,
    load_data,
    get_database_connection
)

In [19]:
import logging

In [7]:
import duckdb

In [22]:
def run_pipeline(input_path,output_path):
    logging.info("Extracting data")
    raw_df = extract_data(input_path)
    logging.info("Validating data")
    validate_data(raw_df)
    logging.info("Cleaning data")
    clean_df = clean_sales_data(raw_df)
    logging.info("Saving CSV")
    load_data(clean_df, output_path)

    logging.info("Opening database connection")
    con = get_database_connection()

    try:
        logging.info("Creating clean_orders table")
        con.execute("""
            CREATE OR REPLACE TABLE clean_orders AS
                SELECT 
                *
                FROM clean_df
        """)
        logging.info("Creating product_summary table")   
        con.execute("""
            CREATE OR REPLACE TABLE product_summary AS    
                SELECT
                    product,
                    SUM(total_sales) AS total_sales,
                    SUM(quantity) AS total_quantity,
                    COUNT(order_id) AS order_count
                FROM clean_orders
                GROUP BY product   
        """)

    except Exception as e:
        logging.error(f"Database pipeline failed: {e}")
        raise    

    finally:
        con.close()

    logging.info("Pipeline completed")    
    return clean_df

In [9]:
input_path = project_root / "data/day_06_pipeline_output.csv"

In [11]:
output_path = project_root / "data/day_13_pipeline_output.csv"

In [12]:
result_df = run_pipeline(input_path,output_path)

In [13]:
result_df.head()

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


In [14]:
output_path.exists()

True

In [15]:
con = get_database_connection()

In [16]:
product_summary_df = con.execute("""
    SELECT
        *
    FROM product_summary
""").df()

In [17]:
display(product_summary_df)

,product,total_sales,total_quantity,order_count
0,Laptop,2400.0,2.0,2
1,Monitor,1500.0,5.0,2
2,Keyboard,100.0,1.0,1


In [18]:
con.close()

Hour 3: Pipeline Status, Failures & Logging

In [23]:
result_df = run_pipeline(input_path, output_path)

2026-09-09 21:44:04,523 - INFO - Extracting data
2026-09-09 21:44:04,526 - INFO - Validating data
2026-09-09 21:44:04,527 - INFO - Cleaning data
2026-09-09 21:44:04,528 - INFO - Saving CSV
2026-09-09 21:44:04,531 - INFO - Opening database connection
2026-09-09 21:44:04,542 - INFO - Creating clean_orders table
2026-09-09 21:44:04,548 - INFO - Creating product_summary table
2026-09-09 21:44:04,564 - INFO - Pipeline completed
